# 06 · Checkpoint: a chat loop with a fake model
Level L1 · Part 1 · core · ~60 min · builds on 01, 02, 03, 04, 05

## Goal
Build a command-line chat loop for Scout over a scripted fake model. The loop keeps the conversation in a
list, counts tokens every turn, never sends more than the context window, and you can show, with printed
output, exactly why Scout "forgets" when the history is not resent.

## The idea
This checkpoint puts Part 1 together. A model is a function: messages in, text out (lesson 01). It writes
tokens until a stop condition (lesson 02), counts everything in tokens (lesson 03), and must fit in a
context window (lesson 05). Our fake model is deterministic, like temperature 0 (lesson 04).

The key fact: the model keeps **nothing** between calls. A chat only feels like a conversation because
your loop stores every message in a list and sends the whole list again each turn.
Stop resending it, and the model has never heard of your project.

Analogy: phoning a help desk where a different person answers every time. You only get continuity if
you read them the case notes at the start of each call.

## Picture
```mermaid
flowchart LR
    U[User types] --> L[Chat loop]
    L -->|append| H[(history list)]
    H -->|"whole list, trimmed to the window"| M[Fake model]
    M -->|reply + token counts| L
    L -->|append reply| H
    L --> P[Print reply]
```

## Requirements
Your chat loop must:
- [ ] read user lines from any iterable of strings (a list now, the keyboard later) and stop on `/quit`;
- [ ] keep the whole conversation in one `history` list that starts with the system message;
- [ ] send the history to the model each turn, or, with `resend_history=False`, only the system message and the latest line;
- [ ] never send more than the window: trim with lesson 05's `sliding_window` and **print a warning** when it drops messages;
- [ ] print each reply with its input tokens, output tokens and stop reason;
- [ ] run as a real command-line program (`python scout_chat.py`), reading from the keyboard or a pipe.

Done when: with history, Scout answers "what is my project about?" correctly; without history, it says
it does not know; and `check_chat_loop(chat_loop)` (below) prints `all checks passed`.

## Build it
### Starter code: Scout's notes and the token counter
These are lesson 01's notes and lesson 05's toy token counter (words and punctuation, plus 1 token per
message for its role), unchanged. Run this cell as is.

In [1]:
import re

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
WINDOW, MAX_OUTPUT = 200, 40     # lesson 05's toy context window and reserved output

def word_tokenize(text: str) -> list[str]:
    return re.findall(r"[\w']+|[.,!?]", text)

def count_tokens(messages: list[dict]) -> int:
    return sum(1 + len(word_tokenize(m["text"])) for m in messages)

SYSTEM = {"role": "system", "text": "You are Scout. Answer only from the user's notes and this chat."}
print(len(NOTES), "notes; system prompt is", count_tokens([SYSTEM]), "tokens")

6 notes; system prompt is 15 tokens


### Starter code: lesson 05's sliding window
Keeps the system message, then as many of the newest messages as fit in `window - max_output` tokens.

In [2]:
def sliding_window(messages: list[dict], window: int, max_output: int) -> list[dict]:
    system, rest = messages[0], messages[1:]
    kept, budget = [], window - max_output - count_tokens([system])
    for m in reversed(rest):                     # newest first
        cost = count_tokens([m])
        if cost > budget:
            break
        kept.insert(0, m)
        budget -= cost
    return [system] + kept

print(len(sliding_window([SYSTEM, {"role": "user", "text": "Hi"}], WINDOW, MAX_OUTPUT)), "messages kept")

2 messages kept


### Starter code: the fake model
`FakeChatModel` is scripted and **stateless**: each reply depends only on the `messages` passed in.
It refuses input over the window (lesson 05), cuts long replies at `max_output` (lesson 02's `max_tokens`),
and returns token counts like a real API's `usage`. Its rules for the reply:
- "My project is ..." gets an acknowledgement;
- a question about "my project" is answered from an earlier user message, if one is in `messages`;
- anything else is answered with the note that shares the most words with the question.

In [3]:
class ContextWindowExceeded(Exception):
    pass

STOPWORDS = {"the", "what", "which", "how", "are", "was", "tell", "about", "me", "is", "a", "on", "in"}

def best_note(question: str) -> dict | None:
    words = {w.lower() for w in word_tokenize(question)} - STOPWORDS
    score = lambda n: len(words & {w.lower() for w in word_tokenize(n["text"])})
    top = max(NOTES, key=score)
    return top if score(top) > 0 else None

def scripted_reply(messages: list[dict]) -> str:
    user = messages[-1]["text"]
    if "my project is" in user.lower():
        return "Got it. I will keep your project in mind."
    if "my project" in user.lower():
        for m in messages[:-1]:
            if m["role"] == "user" and "My project is" in m["text"]:
                return "Your project is" + m["text"].split("My project is", 1)[1]
        return "I don't know what your project is. Could you tell me?"
    note = best_note(user)
    return f"From note {note['id']}: {note['text']}" if note else "None of your notes cover that."

The model itself wraps those rules with the API-like parts: the window check, the output limit and the token counts.

In [4]:
class FakeChatModel:
    def __init__(self, window: int = WINDOW):
        self.window = window
        self.calls = 0                       # how many requests it has served; it stores nothing else

    def __call__(self, messages: list[dict], max_output: int = MAX_OUTPUT) -> dict:
        self.calls += 1
        if messages[-1]["role"] != "user":
            raise ValueError("the last message must be from the user")
        n_in = count_tokens(messages)
        if n_in > self.window:
            raise ContextWindowExceeded(f"400 prompt is too long: {n_in} tokens > {self.window} maximum")
        tokens = word_tokenize(scripted_reply(messages))
        room = min(max_output, self.window - n_in)
        stop = "end_turn" if len(tokens) <= room else "max_tokens"
        text = scripted_reply(messages) if stop == "end_turn" else " ".join(tokens[:room])
        return {"text": text, "input_tokens": n_in, "output_tokens": min(len(tokens), room), "stop_reason": stop}

model = FakeChatModel()
print(model([SYSTEM, {"role": "user", "text": "What are Saturn's rings made of?"}]))

{'text': "From note 6: Saturn's rings are mostly made of ice.", 'input_tokens': 23, 'output_tokens': 11, 'stop_reason': 'end_turn'}


The reply came from note 6, with 23 input tokens (two messages) and `end_turn`. The test script below
is what your loop must pass. It feeds in a scripted conversation and checks the printed result.

In [5]:
import contextlib, io

TEST_INPUTS = [
    "Hi Scout. My project is a school talk about the moons of Mars.",
    "How long is a day on Mars?",
    "Which rover landed on Mars in 2021?",
    "Remind me: what is my project about?",
    "/quit",
    "this line comes after /quit and must never be sent",
]

def check_chat_loop(chat_loop) -> None:
    quiet = io.StringIO()
    with contextlib.redirect_stdout(quiet):            # keep the checks' chat output off screen
        full = chat_loop(FakeChatModel(), TEST_INPUTS)
        forgetful = chat_loop(FakeChatModel(), TEST_INPUTS, resend_history=False)
        chat_loop(FakeChatModel(window=110), TEST_INPUTS, window=110)   # must trim, never raise
    assert full[0] == SYSTEM, "history must start with the system message"
    assert len(full) == 1 + 2 * 4, f"expected 4 user/assistant pairs, got {len(full) - 1} messages"
    assert full[-1]["text"].startswith("Your project is"), "with history Scout should remember"
    assert "don't know" in forgetful[-1]["text"], "without history Scout should not remember"
    assert "dropped" in quiet.getvalue(), "trimming must print a warning"
    assert "never be sent" not in quiet.getvalue(), "the loop must stop at /quit"
    print("all checks passed")

### Your turn
Write `chat_loop(model, inputs, resend_history=True, window=WINDOW, max_output=MAX_OUTPUT)` that returns
the history list, then run `check_chat_loop(chat_loop)`. After that, write `scout_chat.py`, a program that
reads lines from the keyboard (`input()`) and uses your loop.

In [6]:
# your code here

## Reference solution
Try it yourself first. The steps below are one way to meet every requirement.

### R1. The chat loop
Each turn: append the user line, choose what to send, trim it to the window (warning if anything is
dropped), call the model, append its reply. The `history` list is the only memory in the whole system.

In [7]:
def chat_loop(model, inputs, resend_history=True, window=WINDOW, max_output=MAX_OUTPUT, echo=True):
    history = [SYSTEM]
    for user_text in inputs:
        if user_text.strip() == "/quit":
            print("scout> Bye.")
            break
        history.append({"role": "user", "text": user_text})
        to_send = history if resend_history else [SYSTEM, history[-1]]
        trimmed = sliding_window(to_send, window, max_output)
        if len(trimmed) < len(to_send):
            print(f"  ! dropped {len(to_send) - len(trimmed)} old messages to fit the {window}-token window")
        reply = model(trimmed, max_output=max_output)
        history.append({"role": "assistant", "text": reply["text"]})
        if echo:
            print(f"you>   {user_text}")
        print(f"scout> {reply['text']}")
        print(f"       [{reply['input_tokens']} in, {reply['output_tokens']} out, {reply['stop_reason']}]")
    return history

### R2. A chat that remembers
Run the test conversation with the full history resent each turn. Watch the input tokens grow.

In [8]:
history = chat_loop(FakeChatModel(), TEST_INPUTS)
print(len(history), "messages in history")

you>   Hi Scout. My project is a school talk about the moons of Mars.
scout> Got it. I will keep your project in mind.
       [31 in, 11 out, end_turn]
you>   How long is a day on Mars?
scout> From note 3: A day on Mars lasts about 24 hours and 39 minutes.
       [52 in, 15 out, end_turn]
you>   Which rover landed on Mars in 2021?
scout> From note 5: The rover Perseverance landed on Mars in February 2021.
       [77 in, 13 out, end_turn]
you>   Remind me: what is my project about?
scout> Your project is a school talk about the moons of Mars.
       [100 in, 12 out, end_turn]
scout> Bye.
9 messages in history


Turn 4 is answered from turn 1, which was in the request. The input grew from 31 to 100 tokens: every turn
pays again for everything before it. The line after `/quit` was never read.

### R3. Why it forgets
Same model, same lines, but now each request carries only the system message and the newest line.
The loop still stores the history; it just does not send it.

In [9]:
forgetful_model = FakeChatModel()
forgetful = chat_loop(forgetful_model, TEST_INPUTS, resend_history=False)
print(len(forgetful), "messages stored, but the model was only ever shown 2 per call")

you>   Hi Scout. My project is a school talk about the moons of Mars.
scout> Got it. I will keep your project in mind.
       [31 in, 11 out, end_turn]
you>   How long is a day on Mars?
scout> From note 3: A day on Mars lasts about 24 hours and 39 minutes.
       [24 in, 15 out, end_turn]
you>   Which rover landed on Mars in 2021?
scout> From note 5: The rover Perseverance landed on Mars in February 2021.
       [24 in, 13 out, end_turn]
you>   Remind me: what is my project about?
scout> I don't know what your project is. Could you tell me?
       [24 in, 13 out, end_turn]
scout> Bye.
9 messages stored, but the model was only ever shown 2 per call


Scout says it does not know the project. Nothing crashed. The model is not broken: it answered exactly
what it was shown. Let's prove that by replaying the turn-4 request both ways.

In [10]:
turn4_full = history[:-1]                          # system + turns 1-3 + the turn-4 question
turn4_short = [SYSTEM, history[-2]]                # system + the turn-4 question only
for name, msgs in [("full history", turn4_full), ("latest only ", turn4_short)]:
    print(f"{name}: {len(msgs)} messages, {count_tokens(msgs):3} tokens ->", FakeChatModel()(msgs)["text"])

full history: 8 messages, 100 tokens -> Your project is a school talk about the moons of Mars.
latest only : 2 messages,  24 tokens -> I don't know what your project is. Could you tell me?


The answer depends only on the messages in the request. That is what **stateless** means, and it is
true of real model APIs too (lesson 08 shows it with a real model).

### R4. A second way to forget: the window
With a 110-token window, the full history no longer fits from turn 3 on. The loop trims instead of
crashing, and says so.

In [11]:
small = chat_loop(FakeChatModel(window=110), TEST_INPUTS, window=110)

you>   Hi Scout. My project is a school talk about the moons of Mars.
scout> Got it. I will keep your project in mind.
       [31 in, 11 out, end_turn]
you>   How long is a day on Mars?
scout> From note 3: A day on Mars lasts about 24 hours and 39 minutes.
       [52 in, 15 out, end_turn]
  ! dropped 1 old messages to fit the 110-token window
you>   Which rover landed on Mars in 2021?
scout> From note 5: The rover Perseverance landed on Mars in February 2021.
       [61 in, 13 out, end_turn]
  ! dropped 3 old messages to fit the 110-token window
you>   Remind me: what is my project about?
scout> I don't know what your project is. Could you tell me?
       [63 in, 13 out, end_turn]
scout> Bye.


The warning shows the oldest messages, including "My project is...", were dropped, so Scout forgot again,
this time because of the window (lesson 05), not because we skipped resending. Now the acceptance checks:

In [12]:
check_chat_loop(chat_loop)

all checks passed


### R5. A real command-line program
The same loop as a script. `read_lines()` turns the keyboard (or a pipe) into an iterable, so the loop
itself does not change. The model and helpers are copied in so the file runs on its own.

In [13]:
from pathlib import Path
Path("../apps/06").mkdir(parents=True, exist_ok=True)
print("ready to write ../apps/06/scout_chat.py")

ready to write ../apps/06/scout_chat.py


The file below is about 75 lines: `fake_model` and `sliding_window` condensed from the cells above,
`read_lines()` for the keyboard, and `main()` with three flags. In a pipe it echoes each line, since
nobody typed it.

In [14]:
%%writefile ../apps/06/scout_chat.py
"""Scout's first chat loop: python scout_chat.py [--no-history] [--window N]"""
import argparse, re, sys

NOTES = ["Mars has two small moons, Phobos and Deimos.",
         "Venus spins backwards compared with most planets.",
         "A day on Mars lasts about 24 hours and 39 minutes.",
         "Jupiter is the largest planet in the solar system.",
         "The rover Perseverance landed on Mars in February 2021.",
         "Saturn's rings are mostly made of ice."]
SYSTEM = {"role": "system", "text": "You are Scout. Answer only from the user's notes and this chat."}
STOPWORDS = {"the", "what", "which", "how", "are", "was", "tell", "about", "me", "is", "a", "on", "in"}

def tokens(text): return re.findall(r"[\w']+|[.,!?]", text)
def count_tokens(msgs): return sum(1 + len(tokens(m["text"])) for m in msgs)

def fake_model(msgs, window, max_output):
    """Stateless: the reply depends only on msgs."""
    n_in = count_tokens(msgs)
    if n_in > window:
        raise RuntimeError(f"400 prompt is too long: {n_in} tokens > {window} maximum")
    user = msgs[-1]["text"]
    if "my project is" in user.lower():
        text = "Got it. I will keep your project in mind."
    elif "my project" in user.lower():
        said = [m["text"] for m in msgs[:-1] if m["role"] == "user" and "My project is" in m["text"]]
        text = ("Your project is" + said[0].split("My project is", 1)[1]) if said \
            else "I don't know what your project is. Could you tell me?"
    else:
        words = {w.lower() for w in tokens(user)} - STOPWORDS
        scores = [len(words & {w.lower() for w in tokens(n)}) for n in NOTES]
        best = max(range(len(NOTES)), key=scores.__getitem__)
        text = f"From note {best + 1}: {NOTES[best]}" if scores[best] else "None of your notes cover that."
    out = tokens(text)
    room = min(max_output, window - n_in)
    stop = "end_turn" if len(out) <= room else "max_tokens"
    return text if stop == "end_turn" else " ".join(out[:room]), n_in, min(len(out), room), stop

def sliding_window(msgs, window, max_output):
    kept, budget = [], window - max_output - count_tokens([msgs[0]])
    for m in reversed(msgs[1:]):
        if count_tokens([m]) > budget:
            break
        kept.insert(0, m)
        budget -= count_tokens([m])
    return [msgs[0]] + kept

def read_lines(prompt):
    while True:
        try:
            yield input(prompt)
        except EOFError:                     # Ctrl-D, or the end of a pipe
            return

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--no-history", action="store_true")
    ap.add_argument("--window", type=int, default=200)
    ap.add_argument("--max-output", type=int, default=40)
    args = ap.parse_args()
    piped = not sys.stdin.isatty()
    history = [SYSTEM]
    for line in read_lines("" if piped else "you> "):
        if line.strip() == "/quit":
            break
        history.append({"role": "user", "text": line})
        to_send = [SYSTEM, history[-1]] if args.no_history else history
        trimmed = sliding_window(to_send, args.window, args.max_output)
        if len(trimmed) < len(to_send):
            print(f"  ! dropped {len(to_send) - len(trimmed)} old messages to fit the window")
        text, n_in, n_out, stop = fake_model(trimmed, args.window, args.max_output)
        history.append({"role": "assistant", "text": text})
        print(f"{'you>   ' + line + chr(10) if piped else ''}scout> {text}  [{n_in} in, {n_out} out, {stop}]")
    print(f"scout> Bye. ({len(history)} messages in history)")

if __name__ == "__main__":
    main()

Overwriting ../apps/06/scout_chat.py


Run it the way a terminal would, piping the test lines into its standard input, once with history and
once without. In your own terminal, run `python apps/06/scout_chat.py` and type.

In [15]:
import subprocess, sys

session = "\n".join(TEST_INPUTS[:5]) + "\n"
for flags in [[], ["--no-history"]]:
    run = subprocess.run([sys.executable, "../apps/06/scout_chat.py", *flags],
                         input=session, capture_output=True, text=True, timeout=30)
    print(f"--- python scout_chat.py {' '.join(flags)}".rstrip())
    print("\n".join(run.stdout.splitlines()[-3:]))   # turn 4 and the goodbye line

--- python scout_chat.py
you>   Remind me: what is my project about?
scout> Your project is a school talk about the moons of Mars.  [100 in, 12 out, end_turn]
scout> Bye. (9 messages in history)


--- python scout_chat.py --no-history
you>   Remind me: what is my project about?
scout> I don't know what your project is. Could you tell me?  [24 in, 13 out, end_turn]
scout> Bye. (9 messages in history)


The program gives the same two answers as the notebook: it remembers with history and forgets without.

## At work
- **Real chat APIs are stateless too.** Anthropic Messages, Chat Completions and Ollama expect you to
  resend the history each turn; OpenAI's `previous_response_id` and Gemini's `previous_interaction_id`
  keep it on the server instead, which changes what the provider stores. → lessons 08 and 09
- **Resending history makes long chats expensive.** Every turn pays again for all earlier tokens, so teams
  read `usage` on every reply and track cost per turn. → lesson 12
- **Chat loops are unit-tested with a scripted fake model**, exactly like `check_chat_loop`, so tests need
  no key and never flake. → lesson 46
- **One history per user, never shared.** A server runs many chats at once; mixing two users' lists leaks
  one person's data into another's answers. → lessons 38 and 50

## What just happened
- `chat_loop` stored every message in `history` and sent all of it each turn: input grew 31 → 100 tokens, and turn 4 was answered from turn 1.
- With `resend_history=False` the same model said "I don't know": it only ever saw 2 messages per call.
- Replaying the turn-4 request both ways proved the reply depends only on the messages sent: the model is stateless.
- With a 110-token window, `sliding_window` dropped the oldest messages and printed a warning; Scout forgot for a second reason.
- `scout_chat.py` ran the same loop as a real program over a pipe, with the same two results.

## Common mistakes
**1. A mutable default for the history.** This helper looks like it starts a fresh chat for each new caller.
But Python creates `history=[]` once, when the function is defined, so every caller shares one list.

In [16]:
def chat_once(model, user_text, history=[]):          # one list shared by every call!
    if not history:
        history.append(SYSTEM)
    history.append({"role": "user", "text": user_text})
    reply = model(history)
    history.append({"role": "assistant", "text": reply["text"]})
    return reply["text"]

User A tells Scout their project. Then user B, in a "new" chat, asks about theirs.

In [17]:
print("A:", chat_once(FakeChatModel(), "My project is a secret talk about Venus."))
answer_b = chat_once(FakeChatModel(), "Remind me: what is my project about?")
assert "don't know" in answer_b, f"user B was told user A's project: {answer_b!r}"

A: Got it. I will keep your project in mind.


AssertionError: user B was told user A's project: 'Your project is a secret talk about Venus.'

Fix: default to `None` and create a new list inside, or better, keep one history per session id.

In [18]:
SESSIONS: dict[str, list[dict]] = {}

def chat_turn(model, session_id: str, user_text: str) -> str:
    history = SESSIONS.setdefault(session_id, [SYSTEM])
    history.append({"role": "user", "text": user_text})
    reply = model(history)
    history.append({"role": "assistant", "text": reply["text"]})
    return reply["text"]

print("A:", chat_turn(FakeChatModel(), "user-a", "My project is a secret talk about Venus."))
print("B:", chat_turn(FakeChatModel(), "user-b", "Remind me: what is my project about?"))

A: Got it. I will keep your project in mind.
B: I don't know what your project is. Could you tell me?


## Try it
Add two commands to `chat_loop` (in a copy called `chat_loop_v2`): `/reset` empties the history back to just
the system message, and `/history` prints how many messages and tokens the history holds. Show that after
`/reset`, Scout no longer knows the project.

In [19]:
# your code here

## Key terms
- **chat loop** — a loop that reads a user line, appends it to the history, sends the history to the model, appends the reply and repeats until the user quits.
- **conversation history** — the list of messages your code keeps and resends each turn; it is the only memory a chat has.
- **stateless model** — a model whose reply depends only on the messages in the current request; it remembers nothing between calls.
- **checkpoint** — a lesson where you build something from everything in the Part so far, with requirements, a test and a reference solution.